In [1]:
# @title 1. Install Dependencies
!pip uninstall -y gradio huggingface-hub > /dev/null 2>&1
!pip install -q esm@git+https://github.com/Biohub/esm.git@main
!pip install -q py3dmol pandas torch transformers accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
# @title 2. Define Prediction Parameters
COMPLEX_ID = "23527b9b9d26" # @param {type:"string"}
# @markdown Enter amino acid sequences separated by colons (e.g., `SEQ1:SEQ2`):
COLON_SEPARATED_SEQUENCES = "SPWDRVKDLATVYVDVLKDSGRDYVSQFEGSALGKQLKPALNPSVSPSSLKLLDNWDSVTSTFSKLREQLGPVTQEFWDNLEKETEGLRQEMSKDLEEVKAKVQPYLDDFQKKWQEEMELYRQKVEPLRAELQEGARQKLHELQEKLSPLGEEMRDRARAHVDALRTHLAPYSDELRQRLAARLEALKE:QVQLVESGGGLVQAGGSLRLSCAASSKHSVFFHPMGWFRQAPGKEREFVAAVEFFNMYYADSVKGRFTISRDNAKNTVYLQMNSLKPEDTAVYYCVMVDLLYGIYFFHSGSMKWGQGTQVTVSS" # @param {type:"string"}

NUM_ENSEMBLE = 20 # @param {type:"integer"}
ACTIVATION_CHECKPOINTING = True # @param {type:"boolean"}
MSA_MAX_DEPTH = 1024 # @param {type:"integer"}
NUM_LOOPS = 20 # @param {type:"integer"}
NUM_SAMPLING_STEPS = 100 # @param {type:"integer"}
OUTPUT_DIR = "./output_ensemble" # @param {type:"string"}

In [3]:
# @title 3. Import Libraries & Define Utilities
import os
import numpy as np
import pandas as pd
import py3Dmol
from google.colab import files

# Suppress HuggingFace transformers verbosity at the environment level
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

from esm.models.esmfold2 import ProteinInput, StructurePredictionInput, ESMFold2InputBuilder
from transformers.models.esmfold2.modeling_esmfold2 import ESMFold2Model
from torch.distributed.algorithms._checkpoint.checkpoint_wrapper import (
    CheckpointImpl,
    apply_activation_checkpointing,
    checkpoint_wrapper,
)
from transformers.models.esmc.modeling_esmc import UnifiedTransformerBlock as TransformerBlock


def calculate_iplddt(atom_positions, token_to_atoms, chain_ids, plddts, cutoff=4.0):
    """
    Calculates the Interface pLDDT (I-pLDDT) score by computing the average pLDDT
    value for residues within a specified distance cutoff across different chains.
    """
    try:
        coords = np.array(atom_positions)
        cids = np.array(chain_ids)
        plddt_arr = np.array(plddts)

        L = len(cids)
        interface_mask = np.zeros(L, dtype=bool)

        for i in range(L):
            atoms_i = coords[token_to_atoms[i]]
            if len(atoms_i) == 0: continue

            for j in range(i+1, L):
                if cids[i] != cids[j]:
                    atoms_j = coords[token_to_atoms[j]]
                    if len(atoms_j) == 0: continue

                    dist_sq = np.sum((atoms_i[:, None, :] - atoms_j[None, :, :]) ** 2, axis=-1)
                    min_dist = np.sqrt(np.min(dist_sq))

                    if min_dist <= cutoff:
                        interface_mask[i] = True
                        interface_mask[j] = True

        if np.any(interface_mask):
            return float(np.mean(plddt_arr[interface_mask]))
        else:
            return 1.0
    except Exception:
        return 1.0

🚨 No checkpoint found for ESMCForSequenceClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring
🚨 No checkpoint found for ESMCForTokenClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring


In [ ]:
# @title 4. Run Local Structure Prediction
# Prepare output directory
outdir = os.path.abspath(OUTPUT_DIR)
os.makedirs(outdir, exist_ok=True)

import torch

# Parse colon-separated sequences into chains
raw_sequences = COLON_SEPARATED_SEQUENCES.strip().split(":")
chain_letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
sequences = []
for i, seq in enumerate(raw_sequences):
    clean_seq = seq.strip()
    if clean_seq:
        # Assign A, B, C, etc.
        chain_id = chain_letters[i % len(chain_letters)]
        sequences.append((chain_id, clean_seq))

print(f"Parsed {len(sequences)} chain(s) for complex folding.")

print("Loading ESM modules and biohub/ESMFold2 on GPU using Accelerate...")

# Switch to bfloat16 to prevent overflow on -1e9 mask values
model = ESMFold2Model.from_pretrained(
    "biohub/ESMFold2",
    torch_dtype=torch.bfloat16,
    device_map="auto"
).eval()

if ACTIVATION_CHECKPOINTING:
    print("Applying activation checkpointing to TransformerBlock layers to optimize memory...")
    apply_activation_checkpointing(
        model,
        checkpoint_wrapper_fn=lambda m: checkpoint_wrapper(m, checkpoint_impl=CheckpointImpl.NO_REENTRANT),
        check_fn=lambda module: isinstance(module, TransformerBlock),
    )

# Build structure inputs
protein_inputs = [ProteinInput(id=sid, sequence=seq) for sid, seq in sequences]
spi = StructurePredictionInput(sequences=protein_inputs)

results = []
print(f"\nFolding complex '{COMPLEX_ID}' over {NUM_ENSEMBLE} sequential diffusion samples...")

for i in range(NUM_ENSEMBLE):
    print(f"  Generating diffusion sample {i+1}/{NUM_ENSEMBLE}...")

    # Use bfloat16 autocast to safely handle the model's extreme masking values
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        sample_result = ESMFold2InputBuilder().fold(
            model,
            spi,
            num_loops=NUM_LOOPS,
            num_sampling_steps=NUM_SAMPLING_STEPS,
            num_diffusion_samples=1,
            seed=i,
            msa_max_depth=MSA_MAX_DEPTH
        )
    results.append(sample_result)

print(f"\nSelecting and ranking the {NUM_ENSEMBLE} model(s) by pTM...")
results.sort(key=lambda r: float(r.ptm) if r.ptm is not None else -1.0, reverse=True)

csv_data = []
out_csv = os.path.join(outdir, f"{COMPLEX_ID}.csv")

# Extract metrics and save CIF files
for rank, result in enumerate(results, start=1):
    plddt_arr = result.plddt.cpu().numpy() if result.plddt is not None else np.zeros(len(result.complex.sequence))
    plddt = float(plddt_arr.mean())
    ptm = float(result.ptm) if result.ptm is not None else 0.0
    iptm = float(result.iptm) if result.iptm is not None else 0.0

    iplddt = calculate_iplddt(
        atom_positions=result.complex.atom_positions,
        token_to_atoms=result.complex.token_to_atoms,
        chain_ids=result.complex.chain_id,
        plddts=plddt_arr,
        cutoff=4.0
    )

    print(f"Rank {rank} Metrics -> pLDDT: {plddt:.3f}, pTM: {ptm:.3f}, ipTM: {iptm:.3f}, ipLDDT: {iplddt:.3f}")

    raw_mmcif_str = result.complex.to_mmcif()
    suffix = f"_rank{rank}" if NUM_ENSEMBLE > 1 else ""
    out_cif = os.path.join(outdir, f"{COMPLEX_ID}{suffix}.cif")

    with open(out_cif, "w") as f:
        f.write(raw_mmcif_str)

    csv_data.append({
        "query_name": COMPLEX_ID,
        "rank": rank,
        "pLDDT": f"{plddt:.3f}",
        "pTM": f"{ptm:.3f}",
        "ipTM": f"{iptm:.3f}",
        "ipLDDT": f"{iplddt:.3f}",
        "structure_cif": out_cif
    })

df_summary = pd.DataFrame(csv_data)
df_summary.to_csv(out_csv, index=False)
print(f"All tasks complete. Summary saved to '{out_csv}'")

Parsed 2 chain(s) for complex folding.
Loading ESM modules and biohub/ESMFold2 on GPU using Accelerate...


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Applying activation checkpointing to TransformerBlock layers to optimize memory...

Folding complex '23527b9b9d26' over 20 sequential diffusion samples...
  Generating diffusion sample 1/20...
Loading CCD dictionary from /root/.cache/huggingface/hub/models--biohub--ESMFold2/snapshots/8fc3ff471022fdce52c77030685eb775de0c00a3/ccd.pkl
  Generating diffusion sample 2/20...
  Generating diffusion sample 3/20...


In [ ]:
# @title 5. Zip Outputs, Download & Visualize
zip_path = f"/content/{COMPLEX_ID}_ensemble.zip"
!zip -r {zip_path} {OUTPUT_DIR}
files.download(zip_path)

print("\n--- SUMMARY METRICS ---")
display(df_summary)

if len(csv_data) > 0:
    print("\n--- TOP RANKED STRUCTURE ---")
    best_cif_path = csv_data[0]["structure_cif"]
    cif_data_text = open(best_cif_path).read()

    view = py3Dmol.view(width=800, height=500)
    view.addModel(cif_data_text, "mmcif")
    view.setStyle({}, {"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    view.show()
else:
    print("No valid structures were generated to visualize.")